In [ ]:
Objective: To create a simple trading bot which performs fucntions to fetch market data, generate trading signals based on strategy, excute trades in a simulated environment

Outcome: To understand the structure and functionality
live data fetching
implementation strategy
execute trades

In [12]:
from datetime import datetime
import yfinance as yf

In [6]:
class MyTradingStrategy:
    def __init__(self,name):
        self.__name = name
    
    def generate_signal(self,price_data):#use?
        print("this method is intended to be overridden")
        return "Hold"
    @property
    def getter(self):
        return self.__name

    

In [7]:
class MySMATradingStrategy(MyTradingStrategy):
    def __init__(self,lwindow,swindow):
        self.__lwindow = lwindow
        self.__swindow = swindow
        super().__init__("My SMA Trading Strategy")
    @property
    def swindow(self):
        return self.__swindow
    @property
    def lwindow(self):
        return self.__lwindow

    def generate_signal(self, price_data):

        if len(price_data[-self.__lwindow:]) < self.__lwindow:
            return "Hold"
        
        short_avg = sum(price_data[-self.__swindow:])/self.__swindow
        long_avg = sum(price_data[-self.__lwindow:])/self.__lwindow
        
        if short_avg > long_avg:
            return "buy"
        elif short_avg < long_avg:
            return "sell"
        else:
            return "hold"
        

In [8]:
class MyTrade:
    def __init__(self,name,amount,signal):
        self.__name = name
        self.__amount = amount
        self.__signal = signal
        self.__timestamp = datetime.now( )
    @property
    def name(self):
        return self.__name
    @property
    def signal(self):
        return self.__signal
    @property
    def amount(self):
        return self.__amount
    @property
    def timestamp(self):
        return self.__timestamp

    def execute(self):
        print(f"Executed {self.__signal} trade with the strategy {self.__name} at price {self.__amount}")

In [39]:
MySMAObj = MySMATradingStrategy(3,5)

MyBaseObj = MyTrade(MySMAObj.getter,1200,MySMAObj.generate_signal([1,2,3,4,5,5,6,7,8,8,8,9,9,10]))
MyBaseObj.execute()#


Executed sell trade with the strategy My SMA Trading Strategy at price 1200


In [40]:
MyBaseObj.timestamp


datetime.datetime(2026, 9, 9, 2, 14, 50, 758962)

Mock Trading API

In [9]:
class MockTradingAPI:
    def __init__(self, balance):
        self.__balance = balance
    
    @property
    def balance(self):
        return self.__balance
    
    def place_order(self, trade, price):
        if trade.signal == "buy" and price*trade.amount < self.__balance:
            self.__balance -= price*trade.amount
        elif trade.signal == "sell":
            self.__balance += price*trade.amount
        else:
            print("Hold")



In [1]:
pip install yfinance

  Using cached yfinance-1.7.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.16.3-cp310-abi3-win_amd64.whl.metadata (17 kB)
  Using cached lxml-6.1.3-cp312-cp312-win_amd64.whl.metadata (3.4 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached peewee-4.5.1-py3-none-any.whl.metadata (10 kB)
  Using cached protobuf-7.36.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached soupsieve-2.9.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached yfinance-1.7.0-py3-none-any.whl (149 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any

In [20]:

class Tradingsystem:
    def __init__(self, api, strategy_name , symbol):
        self.__api = api
        self.__strategy_name = strategy_name
        self.__symbol = symbol
        self.__price_data = []
    @property
    def api(self):
        return self.__api
    @property
    def strategy_name(self):
        return self.__strategy_name
    @property
    def symbol(self):
        return self.__symbol
    @property
    def price_data(self):
        return self.__price_data
    
    def fetch_price(self):
        data = yf.download(tickers=self.__symbol, period ='1d', interval='1m')
        if not data.empty:
            price = data['Close'].iloc[-1].item()
            self.__price_data.append(price)
            if len(self.__price_data) > self.__strategy_name.lwindow:
                self.__price_data.pop(0)
            print(f"Fetched new price data : {price}")
        else:
            print("No data fetched")
    def run(self):
        self.fetch_price()
        signal = self.__strategy_name.generate_signal(self.__price_data)
        if signal in ['buy','sell']:
            trade = MyTrade(self.__strategy_name, signal,1)
            trade.execute()
            self.__api.place_order(trade,self.__price_data[-1])

In [21]:
if __name__ =="__main__":
    symbol = "AAPL"
    api = MockTradingAPI(balance = 10000)
    strategy = MySMATradingStrategy(3,5)
    system = Tradingsystem(api, strategy,symbol)
    for _ in range(10):
        system.run()
        print(f'Remaining Balance:{api.balance}')


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Fetched new price data : 315.4100036621094
Remaining Balance:10000
Fetched new price data : 315.4100036621094
Remaining Balance:10000



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000
Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000
Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000
Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000
Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000
Fetched new price data : 315.4100036621094
Executed 1 trade with the strategy <__main__.MySMATradingStrategy object at 0x000002166F0AA150> at price sell
Hold
Remaining Balance:10000


Problem Statement<br>Build an extensible algorithmic trading research and backtesting system that compares rule-based and ML-based trading strategies, evaluates them using risk-adjusted performance metrics, and determines whether the strategies generate returns beyond a simple buy-and-hold benchmark after accounting for transaction costs.